In [0]:
%sql
CREATE OR REPLACE TABLE catalog.schema.step7_filtered_cohort_with_apos AS
SELECT
    pe.*,
    CASE WHEN smoke.person_id IS NOT NULL THEN 1 ELSE 0 END AS smoking_status
FROM catalog.schema.step7_filtered_cohort_with_apos pe
LEFT JOIN (
    SELECT DISTINCT person_id
    FROM catalog.schema.drug_exposure_cohort v
    JOIN catalog.omop.concept c ON v.drug_concept_id = c.concept_id
    WHERE c.concept_id = '36249641'

    UNION

    SELECT DISTINCT person_id
    FROM catalog.schema.procedure_occurrence_cohort v
    JOIN catalog.omop.concept c ON v.procedure_concept_id = c.concept_id
    WHERE c.concept_id = '2514534'

    UNION

    SELECT DISTINCT person_id
    FROM catalog.schema.observation_cohort v
    JOIN catalog.omop.concept c ON v.observation_concept_id = c.concept_id
    WHERE c.concept_id IN ('40766945', '903657')
) smoke
    ON pe.person_id = smoke.person_id

Add pre-gravid BMI

In [0]:
%sql
CREATE OR REPLACE TABLE catalog.schema.step7_filtered_cohort_with_apos AS
SELECT
    pe.*,
    COALESCE(bmi.pre_gravid_bmi, -1) AS pre_gravid_bmi
FROM catalog.schema.step7_filtered_cohort_with_apos pe
LEFT JOIN (
    SELECT
        person_id,
        inferred_episode_start,
        recorded_episode_start,
        value_as_number AS pre_gravid_bmi
    FROM (
        SELECT
            v.person_id,
            p.inferred_episode_start,
            p.recorded_episode_start,
            v.value_as_number,
            v.measurement_date,
            ROW_NUMBER() OVER (
                PARTITION BY v.person_id, p.inferred_episode_start, p.recorded_episode_start
                ORDER BY v.measurement_date DESC
            ) AS rn
        FROM catalog.schema.measurement_cohort v
        JOIN catalog.omop.concept c
            ON v.measurement_concept_id = c.concept_id
        JOIN catalog.schema.step7_filtered_cohort_with_apos p
            ON v.person_id = p.person_id
            AND v.measurement_date <= LEAST(p.inferred_episode_start, p.recorded_episode_start)
        WHERE c.concept_id IN (4245997, 44783982)
    )
    WHERE rn = 1
) bmi
    ON pe.person_id = bmi.person_id
    AND pe.inferred_episode_start = bmi.inferred_episode_start
    AND pe.recorded_episode_start = bmi.recorded_episode_start

In [0]:
df = spark.sql("""
    SELECT
        cohort.person_id,
        cohort.age_at_inferred_start,
        cohort.race,
        cohort.ethnicity,
        cohort.smoking_status,
        cohort.pre_gravid_bmi,
        cohort.is_control,
        cohort.has_ghtn,
        cohort.has_gdm,
        cohort.has_pe,
        cohort.has_inferred_pe
    FROM catalog.schema.patient_clmbr_embeddings_new_v5 embeddings
    JOIN catalog.schema.step7_filtered_cohort_with_apos cohort ON cohort.person_id = embeddings.patient_id
""").toPandas()

print(f"Total rows: {len(df)}")

In [0]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------
# 1. Joint PE status: has_pe OR has_inferred_pe counts as PE-positive
# ---------------------------------------------------------------
df['pe_joint'] = ((df['has_pe'] == 1) | (df['has_inferred_pe'] == 1)).astype(int)

# ---------------------------------------------------------------
# 2. Multi-label outcome flags -- a patient can belong to MULTIPLE groups
# ---------------------------------------------------------------
outcome_flags = {
    'Control': 'is_control',
    'GHTN': 'has_ghtn',
    'GDM': 'has_gdm',
    'PE': 'pe_joint',
}

REFERENCE_GROUP = 'Control'
all_groups = list(outcome_flags.keys())
comparison_groups = [g for g in all_groups if g != REFERENCE_GROUP]

continuous_vars = ['age_at_inferred_start', 'pre_gravid_bmi']
categorical_vars = ['race', 'ethnicity', 'smoking_status']

def get_group_members(group_name):
    """Returns the subset of df where that outcome flag == 1."""
    col = outcome_flags[group_name]
    return df[df[col] == 1]

# quick sanity check: group sizes (note these can overlap and won't sum to len(df))
for g in all_groups:
    print(f"{g}: n={len(get_group_members(g))}")

# ---------------------------------------------------------------
# 3. SMD helper functions
# ---------------------------------------------------------------
def smd_continuous(x1, x2):
    m1, m2 = x1.mean(), x2.mean()
    s1, s2 = x1.std(ddof=1), x2.std(ddof=1)
    pooled_sd = np.sqrt((s1**2 + s2**2) / 2)
    if pooled_sd == 0 or np.isnan(pooled_sd):
        return np.nan
    return (m1 - m2) / pooled_sd

def smd_binary(p1, p2):
    pooled_var = (p1 * (1 - p1) + p2 * (1 - p2)) / 2
    if pooled_var == 0:
        return np.nan
    return (p1 - p2) / np.sqrt(pooled_var)

# ---------------------------------------------------------------
# 4. Continuous variable summary: mean, median, missingness, SMD vs Control
# ---------------------------------------------------------------
continuous_rows = []

ref_df = get_group_members(REFERENCE_GROUP)

for var in continuous_vars:
    for group in all_groups:
        sub = get_group_members(group)

        if var == 'pre_gravid_bmi':
            missing_mask = sub[var] == -1
        else:
            missing_mask = sub[var].isna()

        valid = sub.loc[~missing_mask, var]
        n_missing = missing_mask.sum()

        row = {
            'variable': var,
            'outcome_group': group,
            'n_total': len(sub),
            'n_missing': n_missing,
            'n_valid': len(valid),
            'mean': valid.mean() if len(valid) > 0 else np.nan,
            'median': valid.median() if len(valid) > 0 else np.nan,
        }

        if group != REFERENCE_GROUP:
            if var == 'pre_gravid_bmi':
                ref_valid = ref_df.loc[ref_df[var] != -1, var]
            else:
                ref_valid = ref_df.loc[ref_df[var].notna(), var]
            smd = smd_continuous(valid, ref_valid) if len(valid) > 0 and len(ref_valid) > 0 else np.nan
            row['smd_vs_control'] = smd
            row['smd_flag'] = '*' if pd.notna(smd) and abs(smd) > 0.1 else ''
        else:
            row['smd_vs_control'] = np.nan
            row['smd_flag'] = ''

        continuous_rows.append(row)

continuous_summary = pd.DataFrame(continuous_rows)
print("\n=== Continuous variables (multi-label groups) ===")
print(continuous_summary.to_string(index=False))

# ---------------------------------------------------------------
# 5. Categorical variable summary: proportions per category, 
#    missingness, SMD per category vs Control
# ---------------------------------------------------------------
categorical_rows = []

for var in categorical_vars:
    for group in all_groups:
        sub = get_group_members(group)
        missing_mask = sub[var].isna()
        valid = sub.loc[~missing_mask, var]
        n_missing = missing_mask.sum()
        n_valid = len(valid)

        value_counts = valid.value_counts()

        for category, count in value_counts.items():
            p_group = count / n_valid if n_valid > 0 else np.nan

            row = {
                'variable': var,
                'category': category,
                'outcome_group': group,
                'n_total': len(sub),
                'n_missing': n_missing,
                'n_valid': n_valid,
                'n_in_category': count,
                'proportion': p_group,
            }

            if group != REFERENCE_GROUP:
                ref_valid = ref_df.loc[ref_df[var].notna(), var]
                n_ref_valid = len(ref_valid)
                p_ref = (ref_valid == category).sum() / n_ref_valid if n_ref_valid > 0 else np.nan
                smd = smd_binary(p_group, p_ref) if pd.notna(p_group) and pd.notna(p_ref) else np.nan
                row['smd_vs_control'] = smd
                row['smd_flag'] = '*' if pd.notna(smd) and abs(smd) > 0.1 else ''
            else:
                row['smd_vs_control'] = np.nan
                row['smd_flag'] = ''

            categorical_rows.append(row)

        if n_missing > 0:
            categorical_rows.append({
                'variable': var, 'category': 'MISSING', 'outcome_group': group,
                'n_total': len(sub), 'n_missing': n_missing, 'n_valid': n_valid,
                'n_in_category': n_missing, 'proportion': n_missing / len(sub) if len(sub) > 0 else np.nan,
                'smd_vs_control': np.nan, 'smd_flag': ''
            })

categorical_summary = pd.DataFrame(categorical_rows)
print("\n=== Categorical variables (multi-label groups) ===")
print(categorical_summary.to_string(index=False))

# ---------------------------------------------------------------
# 6. Who is missing pre_gravid_bmi / smoking_status, per outcome group
#    (note: a person can now appear under MULTIPLE groups if they have 
#    multiple co-occurring outcomes)
# ---------------------------------------------------------------
print("\n=== Person IDs missing pre_gravid_bmi, by outcome group ===")
for group in all_groups:
    sub = get_group_members(group)
    missing_ids = sub.loc[sub['pre_gravid_bmi'] == -1, 'person_id'].tolist()
    print(f"{group}: {len(missing_ids)} missing -> {missing_ids[:10]}{'...' if len(missing_ids) > 10 else ''}")

print("\n=== Person IDs missing smoking_status, by outcome group ===")
for group in all_groups:
    sub = get_group_members(group)
    missing_ids = sub.loc[sub['smoking_status'].isna(), 'person_id'].tolist()
    print(f"{group}: {len(missing_ids)} missing -> {missing_ids[:10]}{'...' if len(missing_ids) > 10 else ''}")

In [0]:
%pip install tableone --quiet

In [0]:
from tableone import TableOne
import pandas as pd

# ---------------------------------------------------------------
# Reuse the same df already loaded earlier in this notebook 
# (person_id, age_at_inferred_start, race, ethnicity, smoking_status, 
# pre_gravid_bmi, is_control, has_ghtn, has_gdm, has_pe, has_inferred_pe)
# ---------------------------------------------------------------
tableone_df = df.copy()

# pre_gravid_bmi uses -1 as a "missing" sentinel -- convert to a true NA
# so tableone treats it as missing rather than a real, skewing value
tableone_df['pre_gravid_bmi'] = tableone_df['pre_gravid_bmi'].replace(-1, pd.NA)

# simple control vs. case grouping straight from is_control
tableone_df['control_vs_case'] = tableone_df['is_control'].map({1: 'Control', 0: 'Case'})

columns = [
    'age_at_inferred_start',
    'race',
    'ethnicity',
    'smoking_status',
    'pre_gravid_bmi',
]

categorical = ['race', 'ethnicity', 'smoking_status']

table1 = TableOne(
    tableone_df,
    columns=columns,
    categorical=categorical,
    groupby='control_vs_case',
    pval=True,           # include a p-value comparing Control vs Case per variable
    missing=True,
    label_suffix=True,
)

print(table1.tabulate(tablefmt="grid"))

latex_table = table1.to_latex()

# escape LaTeX special characters that tableone doesn't escape itself:
# '%' (comment character) and '_' (subscript character, only valid in math mode)
latex_table = latex_table.replace('%', '\\%').replace('_', '\\_')

print("\n\n=== LaTeX ===\n")
print(latex_table)

with open('/tmp/table_one_control_vs_case.tex', 'w') as f:
    f.write(latex_table)
print("\nSaved to /tmp/table_one_control_vs_case.tex")